In [1]:
# Cell 1 — Notebook Setup (level 5 only)
# This notebook is fully self-contained for level 5. No data is read from or written to any other level's folder.
import os
import pandas as pd
import numpy as np
from typing import Any, Mapping


In [2]:
# Cell 2 — Compile dataset and save to level4_5_logic_gating/data (runnable)
import os
import pandas as pd
from typing import Any, Mapping

# Robust repo root finder (replicated to be self-contained in this cell)
def find_repo_root(start_dir=None):
    d = start_dir or os.getcwd()
    while True:
        if os.path.exists(os.path.join(d, 'requirements.txt')) or os.path.exists(os.path.join(d, '.git')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.getcwd()
        d = parent

repo_root = find_repo_root()
input_path = os.path.join(repo_root, 'level4_5_logic_gating', 'data', 'level5_intents.csv')

# Try both possible column names for intent
# ...rest of cell unchanged, all output and data stays in level4_5_logic_gating/data...


In [3]:
# Cell 3 — Constraint Taxonomy & Governance (declarative, runnable)
# canonical registry of constraints used by level-5 reasoning
CONSTRAINT_REGISTRY = {
    'block_execute': {
        'type': 'hard',
        'description': 'Prevent direct execution when present',
        'rationale': 'Questions/clarifications should not trigger executes'
    },
    'prefer_investigate': {
        'type': 'soft',
        'description': 'Prefer investigation over execution',
        'rationale': 'Uncertainty markers bias toward investigative actions'
    },
    'allow_execute': {
        'type': 'soft',
        'description': 'Allow execution when imperative language is detected',
        'rationale': 'Imperative phrasing indicates operator intent'
    },
    'require_diagnostic_investigate': {
        'type': 'hard',
        'description': 'Require a diagnostic investigation for system-state questions',
        'rationale': 'System-state inquiries require diagnostics before action'
    },
    'require_immediacy_flag': {
        'type': 'soft',
        'description': 'Mark request as immediate/urgent when temporal + imperative',
        'rationale': 'Communicates urgency for downstream handling'
    }
}

# This registry is declarative, importable, and intentionally computation-free.


In [4]:
# Cell 4 — Dataset Sanity & Logical Consistency Checks
import os
import ast
import pandas as pd
from typing import Any

# Load compiled dataset
repo_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(repo_root, 'requirements.txt')) or os.path.exists(os.path.join(repo_root, '.git')):
        break
    repo_root = os.path.dirname(repo_root)
level5_path = os.path.join(repo_root, 'level4_5_logic_gating', 'data', 'level5_intents.csv')
if not os.path.exists(level5_path):
    raise FileNotFoundError(level5_path)

df = pd.read_csv(level5_path)

# Safe parser for list-like cells
def _parse_list_cell(x: Any):
    if isinstance(x, (list, tuple)):
        return list(x)
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x == '':
            return []
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, tuple)):
                return list(val)
        except Exception:
            return [x]
    return []

# Normalize expected list-columns
for col in ['allowed_intents', 'suppressed_intents']:
    df[col] = df[col].apply(_parse_list_cell)

print(f"Loaded {len(df)} rows from {level5_path}")
print(f"Columns: {list(df.columns)}")


Loaded 614 rows from C:\git\nsai_poc\level4_5_logic_gating\data\level5_intents.csv
Columns: ['utterance', 'gold_intent', 'facts', 'active_constraints', 'allowed_intents', 'suppressed_intents']


In [5]:
# Cell 5 — Diagnostics & Sanity Checks
import ast

# Ensure list columns are parsed (in case Cell 4 was re-run standalone)
def _parse_list_cell(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == '':
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return list(v)
    except Exception:
        pass
    return [item.strip() for item in s.split(',') if item.strip()]

for col in ['allowed_intents', 'suppressed_intents']:
    df[col] = df[col].apply(_parse_list_cell)

total = len(df)

# Missing values — use actual columns present in the CSV
check_cols = [c for c in ['utterance', 'gold_intent', 'facts', 'active_constraints', 'allowed_intents', 'suppressed_intents'] if c in df.columns]
missing_counts = df[check_cols].isnull().sum()
print("Missing values per column:")
print(missing_counts.to_string())
print()

# Intent distribution
print("Gold intent distribution:")
print(df['gold_intent'].value_counts().to_string())
print()

# Suppression stats
n_suppressed = df['suppressed_intents'].apply(lambda x: len(x) > 0).sum()
pct_suppressed = round(100 * n_suppressed / total, 1)
print(f"Rows with suppressed intents: {n_suppressed} / {total} ({pct_suppressed}%)")

# Hard constraint: any row where suppressed_intents is non-empty counts as hard-constrained
n_hard = n_suppressed
pct_hard = pct_suppressed
print(f"Rows with hard constraints:   {n_hard} / {total} ({pct_hard}%)")
print()

# Logical violations: gold_intent appears in suppressed_intents (label should never be suppressed)
def is_violation(row):
    intent = str(row['gold_intent']).strip().lower()
    suppressed = [s.strip().lower() for s in row['suppressed_intents']]
    return intent in suppressed

violations = df.apply(is_violation, axis=1)
total_violations = int(violations.sum())
print(f"Logical violations (gold_intent in suppressed_intents): {total_violations}")
if total_violations > 0:
    print("  Violation rows:")
    print(df[violations][['utterance', 'gold_intent', 'suppressed_intents']].to_string())
else:
    print("  ✓ No violations — labels are consistent with constraints")


Missing values per column:
utterance             0
gold_intent           0
facts                 0
active_constraints    0
allowed_intents       0
suppressed_intents    0

Gold intent distribution:
gold_intent
out_of_scope     169
execution        150
investigate      149
summarization    146

Rows with suppressed intents: 179 / 614 (29.2%)
Rows with hard constraints:   179 / 614 (29.2%)

Logical violations (gold_intent in suppressed_intents): 0
  ✓ No violations — labels are consistent with constraints


In [6]:
# Cell 6 — level-5 Readiness Summary (final, lightweight)
# This cell is now fully self-contained for level 5. No references to level 3 or other folders remain.

# verdict
overdict = 'level-5 DATASET READY' if total_violations == 0 else 'LEVEL-5 DATASET BLOCKED'

# machine-readable summary
summary = {
    'total_records': total,
    'percent_with_hard_constraints': round(pct_hard, 2),
    'percent_with_suppressed_intents': round(pct_suppressed, 2),
    'logical_violations_count': int(total_violations),
    'verdict': overdict
}

# human-readable summary
print('level-5 Readiness Summary')
print('Total records:', total)
print(f"% with hard constraints: {summary['percent_with_hard_constraints']}%")
print(f"% with suppressed intents: {summary['percent_with_suppressed_intents']}%")
print('Logical violations (rowwise):', summary['logical_violations_count'])
print('\nVERDICT:', summary['verdict'])

# expose summary variable for downstream cells
LEVEL5_READINESS_SUMMARY = summary


level-5 Readiness Summary
Total records: 614
% with hard constraints: 29.2%
% with suppressed intents: 29.2%
Logical violations (rowwise): 0

VERDICT: level-5 DATASET READY
